# WSense — pipeline ibrida (paradigma del decoder generativo)

## Perche' cambiamo approccio

Il deck *Neural Visual Compression Evolution* descrive due paradigmi. Il **Cap. 1** e' il
VAE / nonlinear transform coding (analisi, quantizzazione, entropy coding con hyperprior,
sintesi): e' quello che avevamo costruito. Il **Cap. 2** e' il **decoder generativo**:

> "modern learned codecs transmit a tiny, highly compressed visual prompt or a heavily
> downsampled latent representation. Models like TinySR and AdcSR are deployed at the
> receiver's end (the decoder side). They take the highly degraded, compressed,
> low-resolution input and use their learned generative priors to synthesize photorealistic
> textures, recover sharp details, and completely eliminate compression noise and artifacts."

Su **questa** metrica il Cap. 2 vince, e si puo' mostrare con i numeri. Il punteggio e'

    S = w1*PSNR/PSNR_ref + w2*MS-SSIM/MS-SSIM_ref + w3*ln(CR)/ln(CR_ref)

Il rate ha il peso maggiore (0.4 sulla slide "Kaggle Metric"; 1/3 sulla pagina Overview)
ma entra tramite un logaritmo, mentre le due metriche di qualita' entrano linearmente:
**dimezzare il payload vale ~6.6 dB di PSNR** (~4.9 a pesi uguali). Un autoencoder
end-to-end negozia un compromesso fissato in training; il Cap. 2 invece comprime in modo
brutale — massimizzando il termine che pesa di piu' — e poi *ricompra* la qualita' con una
rete di restauro lato decoder.

In piu' elimina la parte piu' difficile da allenare: il modello entropico appreso, che
viene sostituito da AVIF/WebP, gia' allo stato dell'arte. Quello che resta da imparare e'
un problema supervisionato semplice (degradato -> pulito), che converge in fretta.

Misura sui dati di validazione **senza nessuna rete**: AVIF a scale 0.35 / q 50 da' gia'
S~1.06, contro 0.845 del nostro autoencoder allenato. Il restauro parte da li'.

**Dataset da allegare:** la competizione + il dataset con `hybrid_codec.py`. GPU attiva.

## 0. Setup

In [ ]:
import sys, glob, pathlib, os, time, random
try:
    from challenge_common import *
except ImportError:
    found = next((p for pat in ["/kaggle/input/*/challenge_common.py", "/kaggle/input/*/*/challenge_common.py",
                  "/kaggle/input/*/*/*/challenge_common.py", "challenge_common.py"]
                  for p in glob.glob(pat)), None)
    if found is None:
        raise FileNotFoundError("challenge_common.py non trovato")
    sys.path.insert(0, str(pathlib.Path(found).parent))
    from challenge_common import *

_own = next((pathlib.Path(p).parent for pat in ["/kaggle/input/*/hybrid_codec.py",
             "/kaggle/input/*/*/hybrid_codec.py", "/kaggle/input/*/*/*/hybrid_codec.py",
             "hybrid_codec.py"] for p in glob.glob(pat)), None)
if _own is None:
    raise FileNotFoundError("hybrid_codec.py non trovato - allega il dataset")
sys.path.insert(0, str(_own))

def _find_real_data_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        if all(t in dirs for t in ("task1", "task2", "task3")):
            return root
    return None
_dr = _find_real_data_root()
if not _dr:
    raise FileNotFoundError("task1/task2/task3 non trovati")
print("dati:", _dr); configure(data_root=_dr)

import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import hybrid_codec as HC
from hybrid_codec import build_restorer, make_cond, degrade, BACKEND, task_score, EQUAL_WEIGHTS

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| backend:", BACKEND)
assert torch.cuda.is_available(), "attiva la GPU"
DEVICE = "cuda"

## 1. Config

In [ ]:
CROP    = 256
BATCH   = 16
EPOCHS  = 12          # il restauro e' un problema supervisionato semplice: converge presto
LR      = 3e-4
BASE    = 32
BLOCKS  = 2
WORKERS = 4

# Pesi della loss: a ~32 dB, 50*MSE e (1 - MS-SSIM) sono dello stesso ordine, quindi i due
# termini di qualita' della metrica pesano in modo confrontabile invece che uno dominare.
W_MSE, W_MSSSIM = 50.0, 1.0

# Punti operativi campionati in training. Meta' nel regime ad alto rate (Task 1 e 3), meta'
# in quello a basso rate con downscale aggressivo, che e' dove vive il Task 2: 5500 byte per
# un frame da diversi megapixel e' un bitrate bassissimo. La rete deve saper ripulire
# ENTRAMBI i regimi, per questo e' condizionata su (q, scale).
def sample_op(rng):
    if rng.random() < 0.5:
        return 1.0, rng.uniform(35, 80)
    return rng.uniform(0.10, 0.60), rng.uniform(8, 60)

print(f"epoche={EPOCHS} batch={BATCH} crop={CROP} base={BASE}")

## 2. MS-SSIM differenziabile (replica di quella del leaderboard, verificata a 1e-15)

In [ ]:
# ---------------------------------------------------------------------------------------
# MS-SSIM differenziabile — replica ESATTA di challenge_common.ms_ssim (verificata contro
# l'originale a 1e-15 di differenza), ma in torch e derivabile, cosi' puo' entrare nella
# loss. Serve perche' la MS-SSIM vale 1/3 del punteggio e finora la loss era solo MSE:
# stavamo ottimizzando la PSNR e sperando che la MS-SSIM seguisse.
# ---------------------------------------------------------------------------------------
import torch
import torch.nn.functional as F

_DATA_RANGE = 255.0
_MSSSIM_W = torch.tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333])   # Wang et al. 2003


def _gauss_kernel_1d(sigma=1.5, truncate=3.5, device="cpu", dtype=torch.float32):
    radius = int(truncate * sigma + 0.5)          # = 5 -> finestra 11x11, come scipy
    x = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
    k = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    return k / k.sum(), radius


def _blur(a, k1d, radius):
    """Convoluzione gaussiana separabile con padding replicate (= mode='nearest' di scipy).
    a: (B,C,H,W); ogni canale filtrato indipendentemente (groups=C)."""
    B, C, H, W = a.shape
    kv = k1d.view(1, 1, -1, 1).expand(C, 1, -1, 1)
    kh = k1d.view(1, 1, 1, -1).expand(C, 1, 1, -1)
    a = F.pad(a, (0, 0, radius, radius), mode="replicate")
    a = F.conv2d(a, kv, groups=C)
    a = F.pad(a, (radius, radius, 0, 0), mode="replicate")
    return F.conv2d(a, kh, groups=C)


def _lcs(x, y, k1d, radius, crop=5):
    C1 = (0.01 * _DATA_RANGE) ** 2
    C2 = (0.03 * _DATA_RANGE) ** 2
    mx, my = _blur(x, k1d, radius), _blur(y, k1d, radius)
    sxx = _blur(x * x, k1d, radius) - mx * mx
    syy = _blur(y * y, k1d, radius) - my * my
    sxy = _blur(x * y, k1d, radius) - mx * my
    l = (2 * mx * my + C1) / (mx * mx + my * my + C1)
    cs = (2 * sxy + C2) / (sxx + syy + C2)
    if crop:
        l, cs = l[..., crop:-crop, crop:-crop], cs[..., crop:-crop, crop:-crop]
    return l.mean(dim=(2, 3)), cs.mean(dim=(2, 3))     # (B,C) ciascuno


def _down2(a):
    h, w = (a.shape[-2] // 2) * 2, (a.shape[-1] // 2) * 2
    return F.avg_pool2d(a[..., :h, :w], 2)


def ms_ssim_torch(x, y, levels=None):
    """x, y: (B,3,H,W) in [-1,1]. Restituisce la MS-SSIM media (scalare, derivabile)."""
    x = (x.clamp(-1, 1) + 1.0) * 127.5
    y = (y.clamp(-1, 1) + 1.0) * 127.5
    n = min(x.shape[-2], x.shape[-1])
    if levels is None:
        levels = int(max(1, min(5, torch.floor(torch.log2(torch.tensor(n / 11.0))).item() + 1)))
    w = _MSSSIM_W[:levels].to(x.device) / _MSSSIM_W[:levels].sum()
    k1d, radius = _gauss_kernel_1d(device=x.device, dtype=x.dtype)

    out = torch.ones(x.shape[0], x.shape[1], device=x.device, dtype=x.dtype)
    for i in range(levels):
        l, cs = _lcs(x, y, k1d, radius)
        term = cs.clamp(min=1e-8) ** w[i]
        if i == levels - 1:
            term = term * l.clamp(min=1e-8) ** w[i]
        out = out * term
        if i < levels - 1:
            x, y = _down2(x), _down2(y)
    return out.mean()

## 3. Dati — degradazione con il backend VERO

Il codec classico gira dentro i worker del DataLoader, quindi il suo costo si nasconde
dietro la GPU. La rete vede esattamente il tipo di degrado che dovra' ripulire a inferenza,
non un'approssimazione.

In [ ]:
TRAIN_PATHS = []
for t in ("task1", "task2", "task3"):
    try:
        TRAIN_PATHS += load_split(t, "train", verbose=False).path.tolist()
    except Exception as e:
        print(f"  {t}: {e}")
print("immagini di training:", len(TRAIN_PATHS))


class RestoreDataset(Dataset):
    """Ritorna (degradato, pulito, cond) su crop casuali. `seed` cambia a ogni epoca, cosi'
    la rete non rivede mai gli stessi crop agli stessi punti operativi."""
    def __init__(self, paths, crop=CROP, length=None, seed=0):
        self.paths, self.crop = list(paths), crop
        self.length = length or len(paths)
        self.seed = seed

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        rng = random.Random((self.seed << 20) + i)
        nprng = np.random.default_rng((self.seed << 20) + i)
        for _ in range(8):
            img = load_image(self.paths[rng.randrange(len(self.paths))])
            if min(img.shape[:2]) >= 64:
                break
        h, w = img.shape[:2]; p = self.crop
        if h < p or w < p:
            img = np.pad(img, ((0, max(0, p - h)), (0, max(0, p - w)), (0, 0)), mode="reflect")
            h, w = img.shape[:2]
        y0, x0 = nprng.integers(0, h - p + 1), nprng.integers(0, w - p + 1)
        clean = np.ascontiguousarray(img[y0:y0 + p, x0:x0 + p])
        if nprng.random() < 0.5:
            clean = np.ascontiguousarray(clean[:, ::-1])
        scale, q = sample_op(nprng)
        deg = degrade(clean, q, scale)
        to_t = lambda a: torch.from_numpy(a).permute(2, 0, 1).float() / 127.5 - 1.0
        return to_t(deg), to_t(clean), torch.tensor([q / 100.0, scale], dtype=torch.float32)


def epoch_loader(epoch):
    ds = RestoreDataset(TRAIN_PATHS, seed=epoch)
    return DataLoader(ds, batch_size=BATCH, num_workers=WORKERS, drop_last=True,
                      pin_memory=True)

print("steps/epoca:", len(TRAIN_PATHS) // BATCH)

## 4. Training

La colonna da guardare e' **gain**: i dB che la rete aggiunge sopra il backend nudo. Se
resta vicino a zero, la rete non sta imparando niente di utile e c'e' qualcosa che non va.

In [ ]:
net = build_restorer(base=BASE, blocks_per_stage=BLOCKS).to(DEVICE)
print("parametri:", f"{sum(p.numel() for p in net.parameters()):,}")

opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-5)
steps = EPOCHS * (len(TRAIN_PATHS) // BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.15)
scaler = torch.amp.GradScaler("cuda")

hist = []
t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    net.train(); t0 = time.time()
    acc = dict(loss=0.0, pin=0.0, pout=0.0, n=0)
    for deg, clean, cond in epoch_loader(epoch):
        deg, clean, cond = (deg.to(DEVICE, non_blocking=True),
                            clean.to(DEVICE, non_blocking=True),
                            cond.to(DEVICE, non_blocking=True))
        with torch.amp.autocast("cuda"):
            out = net(deg, cond)
        out = out.float()
        mse = F.mse_loss(out, clean)
        ms  = ms_ssim_torch(out, clean)
        loss = W_MSE * mse + W_MSSSIM * (1.0 - ms)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        with torch.no_grad():
            pin  = 10 * torch.log10(4.0 / F.mse_loss(deg, clean).clamp(min=1e-12))
            pout = 10 * torch.log10(4.0 / mse.clamp(min=1e-12))
        b = deg.shape[0]
        acc["loss"] += loss.item() * b; acc["pin"] += pin.item() * b
        acc["pout"] += pout.item() * b; acc["n"] += b
    n = acc["n"]
    row = dict(epoch=epoch, loss=acc["loss"]/n, psnr_in=acc["pin"]/n, psnr_out=acc["pout"]/n)
    row["gain"] = row["psnr_out"] - row["psnr_in"]
    hist.append(row)
    print(f"epoch {epoch:3d}  loss={row['loss']:.4f}  backend={row['psnr_in']:6.2f}dB  "
          f"dopo restauro={row['psnr_out']:6.2f}dB  gain={row['gain']:+.2f}dB  "
          f"{time.time()-t0:.0f}s")
print(f"\ntotale {(time.time()-t_start)/60:.1f} min | "
      f"gain medio ultime 3 epoche: {np.mean([h['gain'] for h in hist[-3:]]):+.2f} dB")

## 5. Salva il checkpoint

In [ ]:
CKPT = str(work_dir() / "wsense_restorer.pth")
torch.save({"restorer": net.state_dict(),
            "arch": {"base": BASE, "blocks_per_stage": BLOCKS},
            "backend": BACKEND}, CKPT)
print("checkpoint:", CKPT, f"({os.path.getsize(CKPT)/1e6:.1f} MB)")
print("Pubblicalo come dataset e allegalo al notebook di submission.")

## 6. Punto operativo — misurato, non indovinato

Qui si decide cosa congelare nella submission: si massimizza **S**, non la PSNR. Tieni
`N_VAL` basso, le metriche scipy a risoluzione nativa sono lente.

In [ ]:
from hybrid_codec import sweep_fast

# Sweep VELOCE: il rate e' misurato esatto sull'immagine intera, la qualita' e' stimata su
# pochi ritagli a piena risoluzione invece che ricostruendo tutta l'immagine per ognuno dei
# 25 punti operativi. Lo sweep pieno faceva ~250 ricostruzioni complete + MS-SSIM di scipy
# a risoluzione nativa: decine di minuti. Misurato su immagini 1024x2048: 2.5x piu' veloce
# gia' senza rete di restauro (con la rete il divario cresce, perche' il restauro a tasselli
# costa in proporzione ai megapixel), stesso vincitore e stesso ordinamento dei primi punti.

N_VAL  = 4
SCALES = (1.0, 0.7, 0.5, 0.35, 0.25)
QS     = (20, 35, 50, 65, 80)
val1 = load_split("task1", "val", n_images=N_VAL, verbose=False).path.tolist()

print("== Task 1 ==")
rows1 = sweep_fast(val1, scales=SCALES, qualities=QS, weights_path=CKPT, device=DEVICE,
                   task="task1")

print("\n== Task 3 (CR sull'intera scena) ==")
g3, _ = load_groups("task3", "val", n_scenes=2, max_per_scene=3, verbose=False)
rows3 = sweep_fast(None, scales=SCALES, qualities=QS, weights_path=CKPT, device=DEVICE,
                   task="task3", groups=g3)

OP = {"task1_scale": rows1[0]["scale"], "task1_quality": rows1[0]["quality"],
      "task3_scale": rows3[0]["scale"], "task3_quality": rows3[0]["quality"],
      "arch": {"base": BASE, "blocks_per_stage": BLOCKS}, "backend": BACKEND}
import json as _json
with open(work_dir() / "wsense_operating_point.json", "w") as f:
    _json.dump(OP, f, indent=2)
print("\n" + _json.dumps(OP, indent=2))
print("\n>>> pubblica wsense_restorer.pth + questo json come dataset per la submission")